In [ ]:
import gffutils
import pyfaidx
from Bio.Seq import Seq
import subprocess
from Bio import SeqIO

from Bio import Phylo

import pandas as pd
import json


In [20]:
def remove_zero_length_branches(tree):
    """
    Removes clades with a branch_length of 0 from a Biopython Tree object.
    This effectively prunes zero-length branches and creates polytomies.
    """
    # Create a list to store clades to be processed (children of zero-length clades)
    clades_to_process = []

    # Iterate through all clades in the tree
    for clade in tree.find_clades():
        # Check if the clade has children and a branch_length of 0
        if clade.branch_length == 0 and clade.clades:
            clades_to_process.append(clade)
    print(f"Removing {len(clades_to_process)} zero-length branches")
    # Process clades with zero-length branches
    for zero_clade in clades_to_process:
        parent = tree.get_path(zero_clade)[-2]  # Get the parent of the zero-length clade
        
        if parent:
            # Remove the zero-length clade from its parent's children
            parent.clades.remove(zero_clade)
            # Add the children of the zero-length clade directly to the parent
            parent.clades.extend(zero_clade.clades)

    return tree

def prune_tree_to_seqs(wgstree, fastaf, pruned_tree_file):
    """
    Prune a tree to only include tips present in the fasta file.
    Args:
        wgstree (str): Path to the input tree file (Newick format).
        fastaf (str): Path to the fasta file containing sequence names to keep.
        pruned_tree_file (str): Path to write the pruned tree.
    """
    seq_names = set(SeqIO.to_dict(SeqIO.parse(fastaf, "fasta")).keys())
    
    # Read the tree
    tree = Phylo.read(wgstree, "newick")
    tips = [c.name for c in tree.get_terminals()]
    # Find tips to prune (not in fasta)
    tips_to_prune = [term for term in tips if term not in seq_names]
    
    # Prune tips
    if(len(tips_to_prune) == len(tips)):
        print(len(tips), len(seq_names), 0)
        exit(1)
    else:
        for tip in tips_to_prune:
            tree.prune(tip)    
        #remove zero-length branches
        nozerotree = remove_zero_length_branches(tree) 

        # Write pruned tree
        print(len(tips), len(seq_names), len(tree.get_terminals()))

        Phylo.write(nozerotree, pruned_tree_file, "newick")
    return(len(tree.get_terminals()))

def filter_seqs_to_tree(fastaf, treefile, output_fasta):
    """
    Filter sequences in a FASTA file to only those present as tips in a tree file.
    Args:
        fastaf (str): Path to input FASTA file.
        treefile (str): Path to tree file (Newick format).
        output_fasta (str): Path to output filtered FASTA file.
    """
    # Get tip names from tree
    tree = Phylo.read(treefile, "newick")
    tip_names = set([term.name for term in tree.get_terminals()])
    # Parse FASTA and filter
    records = [rec for rec in SeqIO.parse(fastaf, "fasta")]
    nrecords = len(records)
    goodrecords = [rec for rec in records if rec.id in tip_names]
    # Write filtered FASTA
    with open(output_fasta, "w") as out:
        SeqIO.write(goodrecords, out, "fasta")

    print(nrecords, len(tip_names), len(goodrecords))
    return(len(goodrecords))

In [21]:
def prune_tree_to_samples(wgstree, samples, pruned_tree_file):
    """
    Prune a tree to only include tips present in the samples file.
    Args:
        wgstree (str): Path to the input tree file (Newick format).
        samples (str): Path to the samples file (one sample name per line).
        pruned_tree_file (str): Path to write the pruned tree.
    """
    with open(samples) as f:
        sample_names = set(line.strip() for line in f if line.strip())
    
    # Read the tree
    tree = Phylo.read(wgstree, "newick")
    tips = [c.name for c in tree.get_terminals()]
    # Find tips to prune (not in samples)
    tips_to_prune = [term for term in tips if term not in sample_names]
    
    # Prune tips
    if(len(tips_to_prune) == len(tips)):
        print(len(tips), len(sample_names), 0)
        exit(1)
    else:
        for tip in tips_to_prune:
            tree.prune(tip)    
        #remove zero-length branches
        nozerotree = remove_zero_length_branches(tree) 

        # Write pruned tree
        print(len(tips), len(sample_names), len(tree.get_terminals()))
        Phylo.write(nozerotree, pruned_tree_file, "newick")
    return(len(tree.get_terminals()))

def filter_seqs_to_samples(fastaf, samples, output_fasta):
    """
    Filter sequences in a FASTA file to only those present in the samples file.
    Args:
        fastaf (str): Path to input FASTA file.
        samples (str): Path to the samples file (one sample name per line).
        output_fasta (str): Path to output filtered FASTA file.
    """
    with open(samples) as f:
        sample_names = set(line.strip() for line in f if line.strip())
    
    # Parse FASTA and filter
    records = [rec for rec in SeqIO.parse(fastaf, "fasta")]
    nrecords = len(records)
    goodrecords = [rec for rec in records if rec.id in sample_names]
    # Write filtered FASTA
    with open(output_fasta, "w") as out:
        SeqIO.write(goodrecords, out, "fasta")

    print(nrecords, len(sample_names), len(goodrecords))
    return(len(goodrecords))

In [22]:
# Run FUBAR using HyPhy on the codon alignment and IQ-TREE treefile
def run_fubar(codontrimf, treefile, fubarout):
    errfile = fubarout + ".err"    
    outfile = fubarout + ".out"    
    with open(fubarout, "w") as errfile, open(outfile, "w") as outfile:
        subprocess.run([
            "hyphy", "fubar",
            "--alignment", codontrimf,
            "--tree", treefile,
            "--output", fubarout
        ], check=True,stderr=errfile, stdout=outfile)


In [23]:
def parse_fubar_table(json_file):
    """
    Reads a fubar output file and parses out the table as a pandas DataFrame.
    Args:
        json_file (str): Path to the JSON file.
    Returns:
        pd.DataFrame: Parsed table as DataFrame.
    """
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    jsontable = data["MLE"]["content"]["0"]
    df = pd.DataFrame(jsontable)
    headers = [h[0] for h in data["MLE"]["headers"]]
    #df.columns = headers
    if(len(df.columns) > len(headers)):
        headers[len(headers):len(df.columns)] = ["x" + str(i) for i in range(len(headers), len(df.columns))]
    df.columns = headers
    df.reset_index(names='codon', inplace=True)
    df['dataset'] = json_file.replace(".json","").split("/")[-1]
    return df


In [34]:
#RUN FUBAR FOR ALL SAMPLES

targets = ["RSVA", "RSVB"]
genelist = geneids
# targets = []
# genelist = []

for target in targets:
    reff = f'refs/{target}.fasta'
    gfff = f'refs/{target}.gff3'
    db = gffutils.create_db(gfff, dbfn=':memory:')
    #run fubar for all genes in genelist
    for gene in genelist:
        print(f"Processing {target} {gene}")
        #codon alignment
        codonf = f"geneseqs/{target}_{gene}_codonaln.fasta"             #nucleotide codon alignment

        #filter tree / sequences to match
        wgstree = f'inputs/{target}_tree.nwk'
        wgstrim = f"geneseqs/{target}_{gene}_tree_pruned.nwk"           #pruned wg tree

        #hyphy outputs
        fubarout = f"hyphy/{target}_{gene}_fubar.json"                  #FUBAR output
        fubercsv = f"hyphy/{target}_{gene}_fubar_table.csv"                  #FUBAR table output
        
        #prune tree to sequences, filter sequences to tree
        prune_tree_to_seqs(wgstree, codonf, wgstrim)
        n = filter_seqs_to_tree(codonf, wgstree, codonf)
        if n > 5:
            run_fubar(codonf, wgstrim, fubarout)
            parse_fubar_table(fubarout).to_csv(fubercsv, index=False)
        else:
            print(f"Skipping {target} {gene}, only {n} sequences")

Processing RSVA NS1
Removing 0 zero-length branches
1936 1724 1692
1724 1936 1692
Processing RSVA NS2
Removing 0 zero-length branches
1936 1713 1682
1713 1936 1682
Processing RSVA N
Removing 0 zero-length branches
1936 1433 1404
1433 1936 1404
Processing RSVA P
Removing 0 zero-length branches
1936 1802 1767
1802 1936 1767
Processing RSVA M
Removing 0 zero-length branches
1936 1934 1899
1934 1936 1899
Processing RSVA SH
Removing 0 zero-length branches
1936 1809 1778
1809 1936 1778
Processing RSVA G
Removing 0 zero-length branches
1936 1586 1573
1586 1936 1573
Processing RSVA F
Removing 0 zero-length branches
1936 1939 1904
1939 1936 1904
Processing RSVA M2-1
Removing 0 zero-length branches
1936 1667 1634
1667 1936 1634
Processing RSVA M2-2
Removing 0 zero-length branches
1936 1951 1916
1951 1936 1916
Processing RSVA L
Removing 0 zero-length branches
1936 1782 1749
1782 1936 1749
Processing RSVB NS1
Removing 0 zero-length branches
1864 1788 1765
1788 1864 1765
Processing RSVB NS2
Removin

In [36]:
#RUN FUBAR FOR PGCOE / RV SAMPLES

targets = ["RSVA", "RSVB"]
genelist = geneids

#outdir = "hyphy_pgcoe"
#samples = "inputs/samples_pgcoe.txt"
dataset = "RV"
#dataset = "PGCOE"


for target in targets:
    outdir = f"hyphy_{dataset}"
    samples = f"inputs/{target}_{dataset}.samples"

    reff = f'refs/{target}.fasta'
    gfff = f'refs/{target}.gff3'
    db = gffutils.create_db(gfff, dbfn=':memory:')
    #run fubar for all genes in genelist
    for gene in genelist:
        print(f"Processing {target} {gene} ({outdir})")

        #codon alignment
        codonf = f"geneseqs/{target}_{gene}_codonaln.fasta"             #nucleotide codon alignment
        codonfsamp = f"geneseqs/{target}_{gene}_codonaln_{dataset}.fasta"   #nucleotide codon alignment

        #parse only samples from pruned tree / sequences
        wgstree = f"geneseqs/{target}_{gene}_tree_pruned.nwk"           #pruned wg tree
        wgstrim = f'geneseqs/{target}_{gene}_tree_pruned_{dataset}.nwk'

        #hyphy outputs
        fubarout = f"hyphy/{target}_{gene}_fubar_{dataset}.json"                  #FUBAR output
        fubarcsv = f"hyphy/{target}_{gene}_fubar_table_{dataset}.csv"                  #FUBAR table output

        #prune tree to sequences, filter sequences to tree
        prune_tree_to_samples(wgstree, samples, wgstrim)
        n = filter_seqs_to_samples(codonf, samples, codonfsamp)
        if n > 5:
            run_fubar(codonfsamp, wgstrim, fubarout)
            parse_fubar_table(fubarout).to_csv(fubarcsv, index=False)
        else:
            print(f"Skipping {target} {gene}, only {n} sequences")


Processing RSVA NS1 (hyphy_RV)
Removing 0 zero-length branches
1692 67 49
1692 67 49
Processing RSVA NS2 (hyphy_RV)
Removing 0 zero-length branches
1682 67 38
1682 67 38
Processing RSVA N (hyphy_RV)
Removing 0 zero-length branches
1404 67 1
1404 67 1
Skipping RSVA N, only 1 sequences
Processing RSVA P (hyphy_RV)
Removing 0 zero-length branches
1767 67 44
1767 67 44
Processing RSVA M (hyphy_RV)
Removing 0 zero-length branches
1899 67 64
1899 67 64
Processing RSVA SH (hyphy_RV)
Removing 0 zero-length branches
1778 67 51
1778 67 51
Processing RSVA G (hyphy_RV)
Removing 0 zero-length branches
1573 67 62
1573 67 62
Processing RSVA F (hyphy_RV)
Removing 0 zero-length branches
1904 67 66
1904 67 66
Processing RSVA M2-1 (hyphy_RV)
Removing 0 zero-length branches
1634 67 32
1634 67 32
Processing RSVA M2-2 (hyphy_RV)
Removing 0 zero-length branches
1916 67 67
1916 67 67
Processing RSVA L (hyphy_RV)
Removing 0 zero-length branches
1749 67 38
1749 67 38
Processing RSVB NS1 (hyphy_RV)
Removing 0 ze